# Project 10: Clustering & EM Algorithm from Scratch

**Chapter:** Unit 4 — Unsupervised Learning, Clustering, Mixture Models

**Goal:** Implement K-means clustering and the EM algorithm for Gaussian Mixture Models. Understand the difference between hard and soft assignments, and when EM converges.

**What you'll build:**
1. K-means clustering (hard assignments)
2. Gaussian Mixture Model with EM (soft assignments)
3. Comparison on the same data

**Rules:** No sklearn (except for data generation comparison). NumPy + matplotlib only.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

np.random.seed(42)
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Generate Cluster Data

In [ ]:
def make_blobs(n=300, centers=None, stds=None, seed=42):
    """Generate 2D Gaussian blobs."""
    rng = np.random.default_rng(seed)
    if centers is None:
        centers = np.array([[0, 0], [4, 4], [8, 1]])
    if stds is None:
        stds = [0.8, 1.2, 0.6]
    
    k = len(centers)
    n_per = n // k
    X_parts, y_parts = [], []
    for i, (c, s) in enumerate(zip(centers, stds)):
        X_parts.append(rng.normal(loc=c, scale=s, size=(n_per, 2)))
        y_parts.append(np.full(n_per, i))
    X = np.vstack(X_parts)
    y = np.concatenate(y_parts)
    idx = rng.permutation(len(X))
    return X[idx], y[idx]


def make_elliptical(n=300, seed=42):
    """Clusters with different shapes — K-means will struggle."""
    rng = np.random.default_rng(seed)
    # Elongated cluster
    cov1 = np.array([[3.0, 1.5], [1.5, 1.0]])
    X1 = rng.multivariate_normal([0, 0], cov1, n // 3)
    # Round cluster
    X2 = rng.multivariate_normal([6, 3], 0.5 * np.eye(2), n // 3)
    # Another elongated
    cov3 = np.array([[0.5, -0.4], [-0.4, 2.0]])
    X3 = rng.multivariate_normal([3, -3], cov3, n // 3)
    X = np.vstack([X1, X2, X3])
    y = np.array([0]*(n//3) + [1]*(n//3) + [2]*(n//3))
    idx = rng.permutation(len(X))
    return X[idx], y[idx]


X_blob, y_blob = make_blobs()
X_ellip, y_ellip = make_elliptical()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.scatter(X_blob[:, 0], X_blob[:, 1], c=y_blob, cmap='Set1', s=20, alpha=0.6)
ax1.set_title('Spherical Blobs (K-means friendly)')
ax2.scatter(X_ellip[:, 0], X_ellip[:, 1], c=y_ellip, cmap='Set1', s=20, alpha=0.6)
ax2.set_title('Elliptical Clusters (needs GMM)')
plt.tight_layout(); plt.show()

## 2. K-Means — Your Turn

### The algorithm

1. Initialize $K$ centroids randomly (pick $K$ data points)
2. **Assign** each point to the nearest centroid: $c_i = \arg\min_k \|\mathbf{x}_i - \boldsymbol{\mu}_k\|^2$
3. **Update** centroids: $\boldsymbol{\mu}_k = \frac{1}{|C_k|} \sum_{i \in C_k} \mathbf{x}_i$
4. Repeat until assignments don't change

In [ ]:
class KMeans:
    def __init__(self, k=3, max_iters=100):
        self.k = k
        self.max_iters = max_iters
        self.centroids = None
        self.labels = None
        self.inertia_history = []
    
    def assign(self, X):
        """Assign each point to the nearest centroid.
        
        Returns:
            labels: array of shape (n,) with cluster indices 0..k-1
        """
        ### YOUR CODE ###
        # For each point, compute distance to each centroid, pick the closest
        raise NotImplementedError("Assign step")
    
    def update(self, X):
        """Update centroids to be the mean of assigned points.
        
        Returns:
            new_centroids: array of shape (k, d)
        """
        ### YOUR CODE ###
        # For each cluster, compute the mean of its points
        raise NotImplementedError("Update step")
    
    def inertia(self, X):
        """Sum of squared distances to assigned centroids."""
        ### YOUR CODE ###
        raise NotImplementedError
    
    def fit(self, X):
        """Run K-means. (Outer loop done for you.)"""
        rng = np.random.default_rng(42)
        idx = rng.choice(len(X), self.k, replace=False)
        self.centroids = X[idx].copy()
        
        for i in range(self.max_iters):
            self.labels = self.assign(X)
            new_centroids = self.update(X)
            self.inertia_history.append(self.inertia(X))
            
            if np.allclose(new_centroids, self.centroids):
                print(f"  Converged at iteration {i+1}")
                break
            self.centroids = new_centroids
        return self
    
    def predict(self, X):
        return self.assign(X)

### Sanity checks

In [ ]:
km_test = KMeans(k=3)
km_test.centroids = np.array([[0, 0], [10, 10], [20, 20]], dtype=float)

# Points clearly belonging to each cluster
X_test = np.array([[0.1, 0.2], [9.9, 10.1], [19.5, 20.5]])
labels = km_test.assign(X_test)
assert list(labels) == [0, 1, 2], f"Wrong assignments: {labels}"
print("OK: assign correct")

# Update centroids
km_test.labels = np.array([0, 0, 1])
new_c = km_test.update(X_test)
assert new_c.shape == (3, 2), f"Expected (3,2), got {new_c.shape}"
expected_c0 = np.mean(X_test[:2], axis=0)
assert np.allclose(new_c[0], expected_c0), f"Centroid 0 should be mean of first 2 points"
print("OK: update correct")

# Inertia should be non-negative
km_test.labels = km_test.assign(X_test)
inertia = km_test.inertia(X_test)
assert inertia >= 0, f"Inertia must be >= 0, got {inertia}"
print(f"OK: Inertia = {inertia:.4f}")

# Full fit should converge
km_full = KMeans(k=3)
km_full.fit(X_blob)
assert len(set(km_full.labels)) == 3, "Should find 3 clusters"
print(f"OK: K-means found 3 clusters in {len(km_full.inertia_history)} iterations")

print("\nAll K-means checks passed!")

In [ ]:
km = KMeans(k=3)
km.fit(X_blob)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.scatter(X_blob[:, 0], X_blob[:, 1], c=km.labels, cmap='Set1', s=20, alpha=0.6)
ax1.scatter(km.centroids[:, 0], km.centroids[:, 1], c='black', marker='X', s=200, zorder=5)
ax1.set_title('K-Means Clustering')
ax2.plot(km.inertia_history, 'bo-')
ax2.set_xlabel('Iteration'); ax2.set_ylabel('Inertia'); ax2.set_title('Inertia over Iterations')
ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Gaussian Mixture Model with EM — Your Turn

### The math

GMM models data as a mixture of $K$ Gaussians:

$$p(\mathbf{x}) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(\mathbf{x} \mid \boldsymbol{\mu}_k, \Sigma_k)$$

**E-step** — compute responsibilities (soft assignments):
$$r_{ik} = \frac{\pi_k \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_k, \Sigma_k)}{\sum_{j} \pi_j \, \mathcal{N}(\mathbf{x}_i \mid \boldsymbol{\mu}_j, \Sigma_j)}$$

**M-step** — update parameters:
$$N_k = \sum_i r_{ik}, \quad \boldsymbol{\mu}_k = \frac{1}{N_k}\sum_i r_{ik} \mathbf{x}_i$$
$$\Sigma_k = \frac{1}{N_k}\sum_i r_{ik}(\mathbf{x}_i - \boldsymbol{\mu}_k)(\mathbf{x}_i - \boldsymbol{\mu}_k)^T, \quad \pi_k = \frac{N_k}{n}$$

In [ ]:
def gaussian_pdf(X, mu, sigma):
    """Multivariate Gaussian PDF. (Done for you.)"""
    d = len(mu)
    diff = X - mu  # (n, d)
    sigma_inv = np.linalg.inv(sigma + 1e-6 * np.eye(d))  # regularize
    det = np.linalg.det(sigma + 1e-6 * np.eye(d))
    exponent = -0.5 * np.sum(diff @ sigma_inv * diff, axis=1)
    norm = 1.0 / (np.sqrt((2 * np.pi)**d * det))
    return norm * np.exp(exponent)

In [ ]:
class GMM:
    def __init__(self, k=3, max_iters=100):
        self.k = k
        self.max_iters = max_iters
        self.means = None
        self.covariances = None
        self.weights = None  # mixing coefficients pi_k
        self.log_likelihood_history = []
    
    def e_step(self, X):
        """Compute responsibilities r[i, k] = P(z_i = k | x_i).
        
        Returns:
            r: array of shape (n, k), each row sums to 1
        """
        ### YOUR CODE ###
        # For each k, compute: weights[k] * gaussian_pdf(X, means[k], covariances[k])
        # Then normalize each row to sum to 1
        raise NotImplementedError("E-step: compute responsibilities")
    
    def m_step(self, X, r):
        """Update parameters given responsibilities.
        
        Updates self.means, self.covariances, self.weights in-place.
        """
        ### YOUR CODE ###
        # N_k = sum of r[:, k]
        # means[k] = weighted mean
        # covariances[k] = weighted covariance
        # weights[k] = N_k / n
        raise NotImplementedError("M-step: update means, covariances, weights")
    
    def log_likelihood(self, X):
        """Compute log-likelihood of data under the model."""
        ### YOUR CODE ###
        # For each point: sum_k pi_k * N(x | mu_k, Sigma_k)
        # Return sum of log of those values
        raise NotImplementedError
    
    def fit(self, X):
        """Run EM. (Outer loop done for you.)"""
        n, d = X.shape
        # Initialize with K-means
        km = KMeans(k=self.k)
        km.fit(X)
        self.means = km.centroids.copy()
        self.covariances = np.array([np.eye(d)] * self.k)
        self.weights = np.ones(self.k) / self.k
        
        for i in range(self.max_iters):
            r = self.e_step(X)
            self.m_step(X, r)
            ll = self.log_likelihood(X)
            self.log_likelihood_history.append(ll)
            
            if i > 0 and abs(ll - self.log_likelihood_history[-2]) < 1e-6:
                print(f"  EM converged at iteration {i+1}")
                break
        return self
    
    def predict(self, X):
        r = self.e_step(X)
        return np.argmax(r, axis=1)

### Sanity checks

In [ ]:
gmm_test = GMM(k=2)
gmm_test.means = np.array([[0, 0], [10, 10]], dtype=float)
gmm_test.covariances = np.array([np.eye(2), np.eye(2)])
gmm_test.weights = np.array([0.5, 0.5])

# E-step: point near [0,0] should have high responsibility for cluster 0
X_e = np.array([[0.1, 0.1], [9.9, 10.0]])
r = gmm_test.e_step(X_e)
assert r.shape == (2, 2), f"Expected (2,2), got {r.shape}"
assert abs(r[0].sum() - 1.0) < 1e-6, "Responsibilities must sum to 1"
assert r[0, 0] > 0.99, f"Point near [0,0] should strongly belong to cluster 0, got {r[0,0]:.4f}"
assert r[1, 1] > 0.99, f"Point near [10,10] should strongly belong to cluster 1, got {r[1,1]:.4f}"
print("OK: E-step correct")

# Log-likelihood should be finite
ll = gmm_test.log_likelihood(X_e)
assert np.isfinite(ll), f"Log-likelihood must be finite, got {ll}"
print(f"OK: Log-likelihood = {ll:.4f}")

# Full fit: log-likelihood should increase
gmm_fit = GMM(k=3)
gmm_fit.fit(X_blob)
assert len(gmm_fit.log_likelihood_history) > 1
assert gmm_fit.log_likelihood_history[-1] >= gmm_fit.log_likelihood_history[0] - 1e-6, \
    "Log-likelihood should not decrease during EM"
print(f"OK: EM log-likelihood increased over {len(gmm_fit.log_likelihood_history)} iterations")

print("\nAll GMM checks passed!")

## 4. Compare K-Means vs GMM

In [ ]:
def draw_ellipse(ax, mean, cov, color, n_std=2.0):
    """Draw confidence ellipse for a 2D Gaussian. (Done for you.)"""
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    w, h = 2 * n_std * np.sqrt(vals)
    ellipse = Ellipse(mean, w, h, angle=angle, fill=False, edgecolor=color, linewidth=2)
    ax.add_patch(ellipse)

In [ ]:
# Run both on the elliptical data
km_e = KMeans(k=3)
km_e.fit(X_ellip)

gmm_e = GMM(k=3)
gmm_e.fit(X_ellip)

colors = ['#e41a1c', '#377eb8', '#4daf4a']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# K-means
ax1.scatter(X_ellip[:, 0], X_ellip[:, 1], c=km_e.labels, cmap='Set1', s=20, alpha=0.5)
ax1.scatter(km_e.centroids[:, 0], km_e.centroids[:, 1], c='black', marker='X', s=200, zorder=5)
ax1.set_title('K-Means (spherical assumption)')

# GMM
ax2.scatter(X_ellip[:, 0], X_ellip[:, 1], c=gmm_e.predict(X_ellip), cmap='Set1', s=20, alpha=0.5)
for k in range(3):
    draw_ellipse(ax2, gmm_e.means[k], gmm_e.covariances[k], colors[k % len(colors)])
ax2.set_title('GMM (learns covariance shape)')

for ax in [ax1, ax2]: ax.set_aspect('equal')
plt.tight_layout(); plt.show()

## 5. Experiments

### A: Elbow method
Run K-means with K = 1 to 8. Plot inertia vs K. Where's the "elbow"?

In [ ]:
inertias = []
for k in range(1, 9):
    km_k = KMeans(k=k)
    km_k.fit(X_blob)
    inertias.append(km_k.inertia_history[-1])

plt.plot(range(1, 9), inertias, 'bo-', markersize=8)
plt.xlabel('K'); plt.ylabel('Inertia'); plt.title('Elbow Method')
plt.grid(True, alpha=0.3); plt.show()

**Your observation:** *(Where is the elbow? Does it match the true K=3?)*

...

### B: Initialization sensitivity
Run K-means 5 times with different random seeds. Do you always get the same result?

In [ ]:
# Your experiment here


## 6. What You Learned

1. K-means assumes spherical clusters. When does this fail?
2. What does the E-step of EM compute? How is it different from K-means' hard assignment?
3. Why is EM guaranteed to increase (or not decrease) the log-likelihood at every step?
4. What's the relationship between K-means and EM? (Hint: K-means is EM with hard assignments and spherical covariances.)

---
**Next project:** Reinforcement Learning — learning from rewards.